# 01 — Ingestão bronze
Lê o carimbo mais recente de cada fonte no volume `raw_files` e grava como tabela Delta na camada bronze, **sem limpeza**.

In [0]:
from pyspark.sql import functions as F

CATALOG = "end_to_end_pipeline"
VOLUME_RAW = f"/Volumes/{CATALOG}/bronze/raw_files"

In [0]:
# Formato de cada fonte:
#   "odata" → objeto com os dados dentro da chave "value" (Banco Central)
#   "lista" → lista de registros direto (IBGE e BrasilAPI)
FONTES = {
    "bcb_pix_municipio":           "odata",
    "bcb_estatisticas_transacoes": "odata",
    "bcb_fraudes_med":             "odata",
    "bcb_chaves_pix":              "odata",
    "ibge_municipios":             "lista",
    "ibge_populacao":              "lista",
    "brasilapi_bancos":            "lista",
}

In [0]:
def carimbo_mais_recente(fonte):
    pastas = dbutils.fs.ls(f"{VOLUME_RAW}/{fonte}")
    carimbos = sorted(p.name.rstrip("/") for p in pastas)
    return carimbos[-1]

# Teste
carimbo_mais_recente("ibge_municipios")

In [0]:
def ler_bruto(fonte, formato, carimbo):
    caminho = f"{VOLUME_RAW}/{fonte}/{carimbo}/"

    df = spark.read.option("multiLine", True).json(caminho)

    if formato == "odata":
        df = (
            df.select(
                F.explode("value").alias("registro"),
                F.col("_metadata.file_path").alias("_arquivo_origem"),
            )
            .select("registro.*", "_arquivo_origem")
        )
    else:
        df = df.select("*", F.col("_metadata.file_path").alias("_arquivo_origem"))

    return (
        df.withColumn("_carimbo_coleta", F.lit(carimbo))
          .withColumn("_ingerido_em", F.current_timestamp())
    )

In [0]:
# Teste
fonte = "bcb_fraudes_med"
df_teste = ler_bruto(fonte, FONTES[fonte], carimbo_mais_recente(fonte))

print("Linhas:", df_teste.count())
df_teste.printSchema()
display(df_teste)

In [0]:
# Gravar as 7 tabelas

for fonte, formato in FONTES.items():
    carimbo = carimbo_mais_recente(fonte)
    tabela = f"{CATALOG}.bronze.{fonte}"

    df = ler_bruto(fonte, formato, carimbo)

    (
        df.write
          .format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable(tabela)
    )

    linhas = spark.table(tabela).count()
    print(f"[OK] {tabela}: {linhas} linhas (carimbo {carimbo})")